# 基于可学习光照先验的轻量级低光照图像增强系统

本 notebook 对应 `task.md` 中的课程设计任务，包含：

- Gamma、CLAHE、Retinex 传统增强方法
- 普通轻量级 U-Net baseline
- 可学习 Gamma Map / Illumination Map 先验估计模块
- 光照引导注意力模块
- LPIA-LightU-Net 主模型
- L1 + SSIM + TV 联合损失
- LOL-v1 训练、测试和消融实验入口
- PSNR、SSIM、亮度统计、过曝比例、推理时间、参数量统计
- Gradio 可视化展示系统

第一次运行建议从上到下执行到“模型结构自检”。数据集放好以后，再开启训练和评价单元。

## 0. 依赖安装

如果环境缺少依赖，取消下面代码中的注释并运行。`pyiqa` 只用于可选 NIQE 指标，安装较慢，不影响主线训练与展示。

In [ ]:
# %pip install torch torchvision pillow matplotlib tqdm pandas scikit-image opencv-python gradio
# 可选：用于 NIQE / BRISQUE 等无参考指标
# %pip install pyiqa

## 1. 导入依赖与全局配置

In [ ]:
import os
import math
import time
import random
import json
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageFilter
import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv as mpl_rgb_to_hsv, hsv_to_rgb as mpl_hsv_to_rgb
from tqdm.auto import tqdm

try:
    import cv2
except ImportError:
    cv2 = None
    print("未检测到 opencv-python，CLAHE 会自动使用 scikit-image 备用实现。")

try:
    from skimage import exposure
    from skimage.color import rgb2hsv, hsv2rgb
    from skimage.filters import gaussian
    from skimage.metrics import peak_signal_noise_ratio, structural_similarity
    SKIMAGE_AVAILABLE = True
except Exception as exc:
    exposure = None
    rgb2hsv = None
    hsv2rgb = None
    gaussian = None
    peak_signal_noise_ratio = None
    structural_similarity = None
    SKIMAGE_AVAILABLE = False
    print(f"scikit-image 不可用，将使用 NumPy/PIL 备用实现。原因: {exc}")

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.unicode_minus"] = False


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


@dataclass
class Config:
    project_dir: Path = Path.cwd()
    data_root: Path = Path.cwd() / "datasets"
    output_dir: Path = Path.cwd() / "outputs"
    checkpoint_dir: Path = Path.cwd() / "outputs" / "checkpoints"
    result_dir: Path = Path.cwd() / "outputs" / "results"
    history_csv: Path = Path.cwd() / "outputs" / "history.csv"
    best_ckpt: Path = Path.cwd() / "outputs" / "checkpoints" / "lpia_lightunet_best.pt"
    seed: int = 42
    patch_size: int = 256
    batch_size: int = 2
    num_workers: int = 0
    epochs: int = 50
    lr: float = 2e-4
    weight_decay: float = 1e-6
    base_channels: int = 16
    gamma_min: float = 0.4
    gamma_max: float = 1.0
    lambda_ssim: float = 0.5
    lambda_tv: float = 0.05
    use_amp: bool = True


CFG = Config()
for folder in [CFG.output_dir, CFG.checkpoint_dir, CFG.result_dir]:
    folder.mkdir(parents=True, exist_ok=True)

set_seed(CFG.seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("工作目录:", CFG.project_dir)
print("默认数据目录:", CFG.data_root)
print("设备:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. 数据集路径识别与图像工具

默认支持常见 LOL-v1 目录结构：

```text
datasets/
  our485/
    low/
    high/
  eval15/
    low/
    high/
```

也兼容 `train/low`, `train/high`, `test/low`, `test/high` 等命名。

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def list_images(folder: Path) -> List[Path]:
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES])


def _match_pairs(low_dir: Path, high_dir: Path) -> List[Tuple[Path, Path]]:
    lows = list_images(low_dir)
    highs = list_images(high_dir)
    if not lows or not highs:
        return []
    high_by_name = {p.name.lower(): p for p in highs}
    high_by_stem = {p.stem.lower(): p for p in highs}
    pairs = []
    for low in lows:
        high = high_by_name.get(low.name.lower()) or high_by_stem.get(low.stem.lower())
        if high is not None:
            pairs.append((low, high))
    return pairs


def find_lol_pairs(root: Optional[Path] = None, split: str = "train") -> List[Tuple[Path, Path]]:
    root = Path(root or CFG.data_root)
    split = split.lower()
    if split in {"train", "training"}:
        candidates = [
            (root / "our485" / "low", root / "our485" / "high"),
            (root / "train" / "low", root / "train" / "high"),
            (root / "Train" / "Low", root / "Train" / "Normal"),
            (root / "train" / "low", root / "train" / "normal"),
        ]
    elif split in {"test", "eval", "val", "validation"}:
        candidates = [
            (root / "eval15" / "low", root / "eval15" / "high"),
            (root / "test" / "low", root / "test" / "high"),
            (root / "val" / "low", root / "val" / "high"),
            (root / "Test" / "Low", root / "Test" / "Normal"),
        ]
    else:
        raise ValueError(f"未知 split: {split}")

    for low_dir, high_dir in candidates:
        pairs = _match_pairs(low_dir, high_dir)
        if pairs:
            print(f"{split} 匹配到 {len(pairs)} 对图像")
            print("low :", low_dir)
            print("high:", high_dir)
            return pairs
    print(f"未在 {root} 下找到 {split} 成对数据。")
    return []


def read_rgb(path: Path) -> np.ndarray:
    img = Image.open(path).convert("RGB")
    return np.asarray(img).astype(np.float32) / 255.0


def save_rgb(path: Path, image: np.ndarray) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    image_u8 = np.clip(image * 255.0, 0, 255).astype(np.uint8)
    Image.fromarray(image_u8).save(path)


def np_to_tensor(image: np.ndarray) -> torch.Tensor:
    image = np.clip(image, 0.0, 1.0).astype(np.float32)
    return torch.from_numpy(image).permute(2, 0, 1)


def tensor_to_np(tensor: torch.Tensor) -> np.ndarray:
    if tensor.ndim == 4:
        tensor = tensor[0]
    image = tensor.detach().float().cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    return image


def show_images(images: Dict[str, np.ndarray], cols: Optional[int] = None, figsize: Optional[Tuple[int, int]] = None) -> None:
    names = list(images.keys())
    cols = cols or len(names)
    rows = math.ceil(len(names) / cols)
    figsize = figsize or (4 * cols, 4 * rows)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax, name in zip(axes, names):
        ax.imshow(np.clip(images[name], 0, 1))
        ax.set_title(name)
        ax.axis("off")
    for ax in axes[len(names):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


train_pairs = find_lol_pairs(CFG.data_root, "train")
test_pairs = find_lol_pairs(CFG.data_root, "test")

## 3. 传统低光照增强方法

In [ ]:
def rgb_to_hsv_safe(image: np.ndarray) -> np.ndarray:
    if SKIMAGE_AVAILABLE:
        return rgb2hsv(image)
    return mpl_rgb_to_hsv(np.clip(image, 0, 1))


def hsv_to_rgb_safe(image: np.ndarray) -> np.ndarray:
    if SKIMAGE_AVAILABLE:
        return hsv2rgb(image)
    return mpl_hsv_to_rgb(np.clip(image, 0, 1))


def equalize_hist_gray(value: np.ndarray, bins: int = 256) -> np.ndarray:
    value = np.clip(value, 0, 1).astype(np.float32)
    hist, bin_edges = np.histogram(value.reshape(-1), bins=bins, range=(0, 1), density=False)
    cdf = hist.cumsum().astype(np.float32)
    if cdf[-1] <= 0:
        return value
    cdf = cdf / cdf[-1]
    return np.interp(value.reshape(-1), bin_edges[:-1], cdf).reshape(value.shape).astype(np.float32)


def adaptive_equalize_value(value: np.ndarray, clip_limit: float = 2.0) -> np.ndarray:
    if SKIMAGE_AVAILABLE:
        return exposure.equalize_adapthist(value, clip_limit=min(clip_limit / 40.0, 0.1)).astype(np.float32)
    return equalize_hist_gray(value)


def gaussian_blur_gray(value: np.ndarray, sigma: float) -> np.ndarray:
    value = np.clip(value, 0, 1).astype(np.float32)
    if SKIMAGE_AVAILABLE:
        return gaussian(value, sigma=sigma, preserve_range=True).astype(np.float32)
    if cv2 is not None:
        return cv2.GaussianBlur(value, ksize=(0, 0), sigmaX=float(sigma), sigmaY=float(sigma)).astype(np.float32)
    value_u8 = (value * 255).astype(np.uint8)
    blurred = Image.fromarray(value_u8).filter(ImageFilter.GaussianBlur(radius=float(sigma)))
    return np.asarray(blurred).astype(np.float32) / 255.0


def gamma_correction(image: np.ndarray, gamma: float = 0.6) -> np.ndarray:
    image = np.clip(image, 0.0, 1.0).astype(np.float32)
    return np.power(image + 1e-6, gamma).clip(0, 1)


def clahe_enhance(image: np.ndarray, clip_limit: float = 2.0, tile_grid_size: Tuple[int, int] = (8, 8)) -> np.ndarray:
    image = np.clip(image, 0.0, 1.0).astype(np.float32)
    if cv2 is not None:
        img_u8 = (image * 255).astype(np.uint8)
        lab = cv2.cvtColor(img_u8, cv2.COLOR_RGB2LAB)
        l_channel, a_channel, b_channel = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
        l_enhanced = clahe.apply(l_channel)
        merged = cv2.merge([l_enhanced, a_channel, b_channel])
        rgb = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)
        return rgb.astype(np.float32) / 255.0

    hsv = rgb_to_hsv_safe(image)
    hsv[..., 2] = adaptive_equalize_value(hsv[..., 2], clip_limit=clip_limit)
    return hsv_to_rgb_safe(hsv).astype(np.float32).clip(0, 1)


def percentile_normalize(image: np.ndarray, low: float = 1.0, high: float = 99.0) -> np.ndarray:
    lo, hi = np.percentile(image, (low, high))
    if hi - lo < 1e-8:
        return np.zeros_like(image, dtype=np.float32)
    return np.clip((image - lo) / (hi - lo), 0, 1).astype(np.float32)


def retinex_enhance(image: np.ndarray, sigmas: Sequence[float] = (15, 80, 250), strength: float = 0.85) -> np.ndarray:
    # Multi-scale Retinex on HSV value channel, stable enough for baseline comparison.
    image = np.clip(image, 0.0, 1.0).astype(np.float32)
    hsv = rgb_to_hsv_safe(image)
    value = hsv[..., 2]
    eps = 1e-6
    retinex = np.zeros_like(value, dtype=np.float32)
    for sigma in sigmas:
        illumination = gaussian_blur_gray(value, sigma=sigma)
        retinex += np.log(value + eps) - np.log(illumination + eps)
    retinex = retinex / len(sigmas)
    value_retinex = percentile_normalize(retinex)
    hsv[..., 2] = np.clip((1 - strength) * value + strength * value_retinex, 0, 1)
    return hsv_to_rgb_safe(hsv).astype(np.float32).clip(0, 1)


TRADITIONAL_METHODS = {
    "Gamma(0.6)": lambda img: gamma_correction(img, gamma=0.6),
    "CLAHE": clahe_enhance,
    "Retinex": retinex_enhance,
}


def demo_traditional_methods(image_path: Path) -> None:
    low = read_rgb(image_path)
    outputs = {"Low": low}
    for name, fn in TRADITIONAL_METHODS.items():
        outputs[name] = fn(low)
    show_images(outputs)

## 4. LOL-v1 Dataset 与数据增强

In [ ]:
def ensure_min_size(low: Image.Image, high: Image.Image, patch_size: int) -> Tuple[Image.Image, Image.Image]:
    width, height = low.size
    if width >= patch_size and height >= patch_size:
        return low, high
    scale = max(patch_size / max(width, 1), patch_size / max(height, 1))
    new_size = (math.ceil(width * scale), math.ceil(height * scale))
    return low.resize(new_size, Image.BICUBIC), high.resize(new_size, Image.BICUBIC)


def random_crop_pair(low: Image.Image, high: Image.Image, patch_size: int) -> Tuple[Image.Image, Image.Image]:
    low, high = ensure_min_size(low, high, patch_size)
    width, height = low.size
    left = random.randint(0, width - patch_size)
    top = random.randint(0, height - patch_size)
    box = (left, top, left + patch_size, top + patch_size)
    return low.crop(box), high.crop(box)


def augment_pair(low: Image.Image, high: Image.Image) -> Tuple[Image.Image, Image.Image]:
    if random.random() < 0.5:
        low = ImageOps.mirror(low)
        high = ImageOps.mirror(high)
    if random.random() < 0.5:
        low = ImageOps.flip(low)
        high = ImageOps.flip(high)
    k = random.randint(0, 3)
    if k:
        low = low.rotate(90 * k, expand=True)
        high = high.rotate(90 * k, expand=True)
    return low, high


class PairedLowLightDataset(Dataset):
    def __init__(self, pairs: Sequence[Tuple[Path, Path]], patch_size: int = 256, training: bool = True):
        self.pairs = list(pairs)
        self.patch_size = patch_size
        self.training = training

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        low_path, high_path = self.pairs[index]
        low = Image.open(low_path).convert("RGB")
        high = Image.open(high_path).convert("RGB")
        if self.training:
            low, high = random_crop_pair(low, high, self.patch_size)
            low, high = augment_pair(low, high)
        low_np = np.asarray(low).astype(np.float32) / 255.0
        high_np = np.asarray(high).astype(np.float32) / 255.0
        return {
            "low": np_to_tensor(low_np),
            "high": np_to_tensor(high_np),
            "low_path": str(low_path),
            "high_path": str(high_path),
        }


def make_loaders(
    train_pairs: Sequence[Tuple[Path, Path]],
    test_pairs: Sequence[Tuple[Path, Path]],
    batch_size: int = CFG.batch_size,
    patch_size: int = CFG.patch_size,
) -> Tuple[Optional[DataLoader], Optional[DataLoader]]:
    train_loader = None
    test_loader = None
    if train_pairs:
        train_dataset = PairedLowLightDataset(train_pairs, patch_size=patch_size, training=True)
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=CFG.num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=True,
        )
    if test_pairs:
        test_dataset = PairedLowLightDataset(test_pairs, patch_size=patch_size, training=False)
        test_loader = DataLoader(
            test_dataset,
            batch_size=1,
            shuffle=False,
            num_workers=CFG.num_workers,
            pin_memory=torch.cuda.is_available(),
        )
    return train_loader, test_loader

## 5. 模型：LPIA-LightU-Net

结构对应任务书技术路线：

```text
I_low -> Prior Estimator -> Gamma Map / Illumination Map
      -> I_prior = I_low ^ GammaMap
      -> concat(I_low, I_prior, Illumination Map)
      -> Illumination-Guided LightU-Net
      -> I_out
```

In [ ]:
def rgb_to_luminance_tensor(x: torch.Tensor) -> torch.Tensor:
    r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
    return 0.299 * r + 0.587 * g + 0.114 * b


def group_norm(channels: int) -> nn.GroupNorm:
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return nn.GroupNorm(groups, channels)
    return nn.GroupNorm(1, channels)


class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            group_norm(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            group_norm(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class LearnableGammaPrior(nn.Module):
    def __init__(self, gamma_min: float = 0.4, gamma_max: float = 1.0):
        super().__init__()
        self.gamma_min = gamma_min
        self.gamma_max = gamma_max
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        raw = self.net(x)
        gamma = self.gamma_min + (self.gamma_max - self.gamma_min) * torch.sigmoid(raw)
        prior = torch.pow(torch.clamp(x, min=1e-6, max=1.0), gamma)
        luminance = rgb_to_luminance_tensor(x)
        illumination = torch.clamp(0.5 * luminance + 0.5 * gamma, 0.0, 1.0)
        return prior, gamma, illumination


class FixedGammaPrior(nn.Module):
    def __init__(self, gamma: float = 0.6):
        super().__init__()
        self.gamma = gamma

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        gamma_map = torch.full((x.shape[0], 1, x.shape[2], x.shape[3]), self.gamma, device=x.device, dtype=x.dtype)
        prior = torch.pow(torch.clamp(x, min=1e-6, max=1.0), gamma_map)
        luminance = rgb_to_luminance_tensor(x)
        illumination = torch.clamp(0.5 * luminance + 0.5 * gamma_map, 0.0, 1.0)
        return prior, gamma_map, illumination


class IlluminationGuidedAttention(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Conv2d(1, channels, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, feature: torch.Tensor, illumination: torch.Tensor) -> torch.Tensor:
        dark_response = 1.0 - F.interpolate(
            illumination,
            size=feature.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        weight = self.attention(dark_response)
        return feature * weight + feature


class UNetBackbone(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 3, base_channels: int = 16, use_attention: bool = True):
        super().__init__()
        b = base_channels
        self.use_attention = use_attention
        self.pool = nn.MaxPool2d(2)
        self.enc1 = ConvBlock(in_channels, b)
        self.enc2 = ConvBlock(b, b * 2)
        self.enc3 = ConvBlock(b * 2, b * 4)
        self.enc4 = ConvBlock(b * 4, b * 8)
        self.bottleneck = ConvBlock(b * 8, b * 8)
        self.attention = IlluminationGuidedAttention(b * 8) if use_attention else nn.Identity()
        self.up4 = nn.ConvTranspose2d(b * 8, b * 8, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(b * 16, b * 8)
        self.up3 = nn.ConvTranspose2d(b * 8, b * 4, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(b * 8, b * 4)
        self.up2 = nn.ConvTranspose2d(b * 4, b * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(b * 4, b * 2)
        self.up1 = nn.ConvTranspose2d(b * 2, b, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(b * 2, b)
        self.out = nn.Sequential(nn.Conv2d(b, out_channels, kernel_size=1), nn.Sigmoid())

    @staticmethod
    def _concat(up: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        if up.shape[-2:] != skip.shape[-2:]:
            up = F.interpolate(up, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return torch.cat([up, skip], dim=1)

    def forward(self, x: torch.Tensor, illumination: Optional[torch.Tensor] = None) -> torch.Tensor:
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        if self.use_attention and illumination is not None:
            b = self.attention(b, illumination)
        d4 = self.dec4(self._concat(self.up4(b), e4))
        d3 = self.dec3(self._concat(self.up3(d4), e3))
        d2 = self.dec2(self._concat(self.up2(d3), e2))
        d1 = self.dec1(self._concat(self.up1(d2), e1))
        return self.out(d1)


class LPIALightUNet(nn.Module):
    def __init__(
        self,
        prior_mode: str = "learnable",
        use_attention: bool = True,
        base_channels: int = 16,
        gamma_min: float = 0.4,
        gamma_max: float = 1.0,
        fixed_gamma: float = 0.6,
    ):
        super().__init__()
        prior_mode = prior_mode.lower()
        self.prior_mode = prior_mode
        if prior_mode == "learnable":
            self.prior = LearnableGammaPrior(gamma_min=gamma_min, gamma_max=gamma_max)
            in_channels = 7
        elif prior_mode == "fixed":
            self.prior = FixedGammaPrior(gamma=fixed_gamma)
            in_channels = 7
        elif prior_mode == "none":
            self.prior = None
            in_channels = 3
        else:
            raise ValueError("prior_mode 必须是 learnable、fixed 或 none")
        self.backbone = UNetBackbone(
            in_channels=in_channels,
            out_channels=3,
            base_channels=base_channels,
            use_attention=use_attention,
        )

    def forward(self, x: torch.Tensor, return_aux: bool = False):
        aux = {}
        if self.prior is None:
            illumination = rgb_to_luminance_tensor(x)
            model_input = x
            aux["illumination"] = illumination
        else:
            prior, gamma_map, illumination = self.prior(x)
            model_input = torch.cat([x, prior, illumination], dim=1)
            aux.update({"prior": prior, "gamma_map": gamma_map, "illumination": illumination})
        output = self.backbone(model_input, illumination)
        if return_aux:
            return output, aux
        return output


def build_model(variant: str = "lpia_lightunet", base_channels: int = CFG.base_channels) -> LPIALightUNet:
    variant = variant.lower()
    if variant == "plain_unet":
        return LPIALightUNet(prior_mode="none", use_attention=False, base_channels=base_channels)
    if variant == "attention_unet":
        return LPIALightUNet(prior_mode="none", use_attention=True, base_channels=base_channels)
    if variant == "fixed_gamma_unet":
        return LPIALightUNet(prior_mode="fixed", use_attention=False, base_channels=base_channels, fixed_gamma=0.6)
    if variant == "learnable_gamma_unet":
        return LPIALightUNet(
            prior_mode="learnable",
            use_attention=False,
            base_channels=base_channels,
            gamma_min=CFG.gamma_min,
            gamma_max=CFG.gamma_max,
        )
    if variant == "lpia_lightunet":
        return LPIALightUNet(
            prior_mode="learnable",
            use_attention=True,
            base_channels=base_channels,
            gamma_min=CFG.gamma_min,
            gamma_max=CFG.gamma_max,
        )
    raise ValueError(f"未知模型变体: {variant}")


EXPERIMENTS = {
    "plain_unet": "普通轻量级 U-Net，不使用 Gamma 先验和注意力",
    "attention_unet": "光照引导注意力 + 普通轻量级 U-Net，不使用 Gamma 先验",
    "fixed_gamma_unet": "固定 Gamma 先验 + 轻量 U-Net",
    "learnable_gamma_unet": "可学习 Gamma Map + 轻量 U-Net",
    "lpia_lightunet": "可学习 Gamma Map + 光照引导注意力 + 轻量 U-Net",
}

## 6. 损失函数：L1 + SSIM + TV

In [ ]:
def ssim_torch(x: torch.Tensor, y: torch.Tensor, window_size: int = 11) -> torch.Tensor:
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    padding = window_size // 2
    mu_x = F.avg_pool2d(x, window_size, stride=1, padding=padding)
    mu_y = F.avg_pool2d(y, window_size, stride=1, padding=padding)
    sigma_x = F.avg_pool2d(x * x, window_size, stride=1, padding=padding) - mu_x * mu_x
    sigma_y = F.avg_pool2d(y * y, window_size, stride=1, padding=padding) - mu_y * mu_y
    sigma_xy = F.avg_pool2d(x * y, window_size, stride=1, padding=padding) - mu_x * mu_y
    ssim_map = ((2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)) / (
        (mu_x ** 2 + mu_y ** 2 + c1) * (sigma_x + sigma_y + c2) + 1e-8
    )
    return ssim_map.mean()


def tv_loss(x: Optional[torch.Tensor]) -> torch.Tensor:
    if x is None:
        return torch.tensor(0.0, device=DEVICE)
    loss_h = torch.mean(torch.abs(x[:, :, 1:, :] - x[:, :, :-1, :]))
    loss_w = torch.mean(torch.abs(x[:, :, :, 1:] - x[:, :, :, :-1]))
    return loss_h + loss_w


class EnhancementLoss(nn.Module):
    def __init__(self, lambda_ssim: float = 0.5, lambda_tv: float = 0.05):
        super().__init__()
        self.lambda_ssim = lambda_ssim
        self.lambda_tv = lambda_tv
        self.l1 = nn.L1Loss()

    def forward(self, pred: torch.Tensor, target: torch.Tensor, aux: Optional[Dict[str, torch.Tensor]] = None):
        aux = aux or {}
        l1_value = self.l1(pred, target)
        ssim_value = 1.0 - ssim_torch(pred, target)
        tv_value = tv_loss(aux.get("gamma_map"))
        total = l1_value + self.lambda_ssim * ssim_value + self.lambda_tv * tv_value
        logs = {
            "loss": float(total.detach().cpu()),
            "l1": float(l1_value.detach().cpu()),
            "ssim_loss": float(ssim_value.detach().cpu()),
            "tv": float(tv_value.detach().cpu()),
        }
        return total, logs

## 7. 训练与验证函数

In [ ]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def estimate_model_size_mb(model: nn.Module) -> float:
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    return (param_bytes + buffer_bytes) / (1024 ** 2)


def get_amp_tools(enabled: bool):
    enabled = enabled and DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=enabled)
    autocast = torch.cuda.amp.autocast
    return scaler, autocast, enabled


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: EnhancementLoss,
    scaler,
    autocast,
    amp_enabled: bool,
    epoch: int,
) -> Dict[str, float]:
    model.train()
    meters = {"loss": [], "l1": [], "ssim_loss": [], "tv": []}
    pbar = tqdm(loader, desc=f"Train {epoch}", leave=False)
    for batch in pbar:
        low = batch["low"].to(DEVICE, non_blocking=True)
        high = batch["high"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=amp_enabled):
            pred, aux = model(low, return_aux=True)
            loss, logs = criterion(pred, high, aux)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        for key in meters:
            meters[key].append(logs[key])
        pbar.set_postfix({key: f"{np.mean(vals):.4f}" for key, vals in meters.items()})
    return {key: float(np.mean(vals)) for key, vals in meters.items()}


@torch.no_grad()
def validate_one_epoch(model: nn.Module, loader: DataLoader, criterion: EnhancementLoss) -> Dict[str, float]:
    model.eval()
    meters = {"loss": [], "l1": [], "ssim_loss": [], "tv": [], "psnr": [], "ssim": []}
    for batch in tqdm(loader, desc="Validate", leave=False):
        low = batch["low"].to(DEVICE, non_blocking=True)
        high = batch["high"].to(DEVICE, non_blocking=True)
        pred, aux = model(low, return_aux=True)
        loss, logs = criterion(pred, high, aux)
        pred_np = tensor_to_np(pred)
        high_np = tensor_to_np(high)
        psnr_value, ssim_value = full_reference_metrics(pred_np, high_np)
        for key in ["loss", "l1", "ssim_loss", "tv"]:
            meters[key].append(logs[key])
        meters["psnr"].append(psnr_value)
        meters["ssim"].append(ssim_value)
    return {key: float(np.mean(vals)) for key, vals in meters.items()}


def save_checkpoint(path: Path, model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int, metrics: Dict[str, float], variant: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "variant": variant,
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "metrics": metrics,
            "config": {k: str(v) if isinstance(v, Path) else v for k, v in asdict(CFG).items()},
        },
        path,
    )


def load_checkpoint(path: Path, model: Optional[nn.Module] = None, map_location: Optional[torch.device] = None) -> nn.Module:
    path = Path(path)
    ckpt = torch.load(path, map_location=map_location or DEVICE)
    variant = ckpt.get("variant", "lpia_lightunet")
    if model is None:
        model = build_model(variant)
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.to(DEVICE)
    model.eval()
    print(f"已加载 checkpoint: {path}，epoch={ckpt.get('epoch')}，variant={variant}")
    return model


def fit_model(
    variant: str = "lpia_lightunet",
    epochs: int = CFG.epochs,
    checkpoint_path: Path = CFG.best_ckpt,
) -> Tuple[nn.Module, pd.DataFrame]:
    train_loader, val_loader = make_loaders(train_pairs, test_pairs)
    if train_loader is None or val_loader is None:
        raise RuntimeError("没有找到 LOL-v1 训练/测试数据。请先放置数据集或修改 CFG.data_root。")
    model = build_model(variant).to(DEVICE)
    print(model.__class__.__name__, variant)
    print(f"参数量: {count_parameters(model) / 1e6:.3f} M")
    print(f"模型大小估计: {estimate_model_size_mb(model):.2f} MB")
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs), eta_min=CFG.lr * 0.05)
    criterion = EnhancementLoss(lambda_ssim=CFG.lambda_ssim, lambda_tv=CFG.lambda_tv)
    scaler, autocast, amp_enabled = get_amp_tools(CFG.use_amp)
    best_psnr = -float("inf")
    records = []
    for epoch in range(1, epochs + 1):
        train_logs = train_one_epoch(model, train_loader, optimizer, criterion, scaler, autocast, amp_enabled, epoch)
        val_logs = validate_one_epoch(model, val_loader, criterion)
        scheduler.step()
        record = {"epoch": epoch, "lr": optimizer.param_groups[0]["lr"]}
        record.update({f"train_{k}": v for k, v in train_logs.items()})
        record.update({f"val_{k}": v for k, v in val_logs.items()})
        records.append(record)
        history = pd.DataFrame(records)
        history.to_csv(CFG.history_csv, index=False)
        print(
            f"Epoch {epoch:03d}/{epochs} | "
            f"train_loss={record['train_loss']:.4f} | "
            f"val_psnr={record['val_psnr']:.2f} | val_ssim={record['val_ssim']:.4f}"
        )
        if val_logs["psnr"] > best_psnr:
            best_psnr = val_logs["psnr"]
            save_checkpoint(checkpoint_path, model, optimizer, epoch, val_logs, variant)
            print(f"保存最佳模型: {checkpoint_path}")
    return model, pd.DataFrame(records)

## 8. 评价指标与效率统计

In [ ]:
def psnr_np_fallback(pred: np.ndarray, target: np.ndarray) -> float:
    mse = float(np.mean((np.clip(pred, 0, 1) - np.clip(target, 0, 1)) ** 2))
    if mse <= 1e-12:
        return float("inf")
    return 20.0 * math.log10(1.0 / math.sqrt(mse))


def ssim_np_fallback(pred: np.ndarray, target: np.ndarray) -> float:
    # Global channel-wise SSIM fallback. For formal reporting, prefer scikit-image SSIM.
    pred = np.clip(pred, 0, 1).astype(np.float64)
    target = np.clip(target, 0, 1).astype(np.float64)
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    scores = []
    for channel in range(pred.shape[-1]):
        x = pred[..., channel]
        y = target[..., channel]
        mux, muy = x.mean(), y.mean()
        vx, vy = x.var(), y.var()
        cov = ((x - mux) * (y - muy)).mean()
        score = ((2 * mux * muy + c1) * (2 * cov + c2)) / ((mux ** 2 + muy ** 2 + c1) * (vx + vy + c2) + 1e-12)
        scores.append(score)
    return float(np.mean(scores))


def full_reference_metrics(pred: np.ndarray, target: np.ndarray) -> Tuple[float, float]:
    pred = np.clip(pred, 0, 1)
    target = np.clip(target, 0, 1)
    if SKIMAGE_AVAILABLE:
        psnr_value = peak_signal_noise_ratio(target, pred, data_range=1.0)
        ssim_value = structural_similarity(target, pred, channel_axis=-1, data_range=1.0)
    else:
        psnr_value = psnr_np_fallback(pred, target)
        ssim_value = ssim_np_fallback(pred, target)
    return float(psnr_value), float(ssim_value)


def luminance_np(image: np.ndarray) -> np.ndarray:
    return 0.299 * image[..., 0] + 0.587 * image[..., 1] + 0.114 * image[..., 2]


def brightness_metrics(input_img: np.ndarray, output_img: np.ndarray) -> Dict[str, float]:
    y_in = luminance_np(input_img)
    y_out = luminance_np(output_img)
    dark = y_in < 0.3
    mid = (y_in >= 0.3) & (y_in < 0.7)
    bright = y_in >= 0.7

    def mean_delta(mask: np.ndarray) -> float:
        if mask.sum() == 0:
            return float("nan")
        return float((y_out[mask] - y_in[mask]).mean())

    return {
        "input_mean_y": float(y_in.mean()),
        "output_mean_y": float(y_out.mean()),
        "mean_y_gain": float(y_out.mean() - y_in.mean()),
        "dark_y_gain": mean_delta(dark),
        "mid_y_gain": mean_delta(mid),
        "bright_y_gain": mean_delta(bright),
        "over_exposed_ratio": float((y_out > 0.95).mean()),
    }


def calculate_niqe_optional(image: np.ndarray) -> float:
    'Return NIQE if pyiqa is installed, otherwise NaN.'
    try:
        import pyiqa
    except ImportError:
        return float("nan")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    metric = pyiqa.create_metric("niqe", device=device)
    tensor = np_to_tensor(image).unsqueeze(0).to(device)
    with torch.no_grad():
        return float(metric(tensor).detach().cpu())


@torch.no_grad()
def predict_model_np(model: nn.Module, image: np.ndarray, device: torch.device = DEVICE) -> np.ndarray:
    model.eval()
    tensor = np_to_tensor(image).unsqueeze(0).to(device)
    pred = model(tensor)
    return tensor_to_np(pred)


def measure_model_inference_time(
    model: nn.Module,
    image: np.ndarray,
    repeats: int = 20,
    warmup: int = 5,
    device: torch.device = DEVICE,
) -> Dict[str, float]:
    model.eval()
    tensor = np_to_tensor(image).unsqueeze(0).to(device)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(tensor)
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(repeats):
            _ = model(tensor)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
    mean_ms = elapsed / repeats * 1000
    return {"mean_ms": mean_ms, "fps": 1000.0 / max(mean_ms, 1e-8)}


def evaluate_traditional_methods(pairs: Sequence[Tuple[Path, Path]], max_images: Optional[int] = None) -> pd.DataFrame:
    records = []
    selected = list(pairs[:max_images] if max_images else pairs)
    for low_path, high_path in tqdm(selected, desc="Traditional eval"):
        low = read_rgb(low_path)
        high = read_rgb(high_path)
        for method_name, fn in TRADITIONAL_METHODS.items():
            start = time.perf_counter()
            enhanced = fn(low)
            elapsed_ms = (time.perf_counter() - start) * 1000
            psnr_value, ssim_value = full_reference_metrics(enhanced, high)
            record = {
                "image": low_path.name,
                "method": method_name,
                "psnr": psnr_value,
                "ssim": ssim_value,
                "time_ms": elapsed_ms,
                "niqe": calculate_niqe_optional(enhanced),
            }
            record.update(brightness_metrics(low, enhanced))
            records.append(record)
    df = pd.DataFrame(records)
    if not df.empty:
        display(df.groupby("method")[["psnr", "ssim", "time_ms", "mean_y_gain", "dark_y_gain", "over_exposed_ratio"]].mean())
    return df


@torch.no_grad()
def evaluate_deep_model(
    model: nn.Module,
    pairs: Sequence[Tuple[Path, Path]],
    method_name: str = "LPIA-LightU-Net",
    max_images: Optional[int] = None,
) -> pd.DataFrame:
    records = []
    selected = list(pairs[:max_images] if max_images else pairs)
    for low_path, high_path in tqdm(selected, desc=f"{method_name} eval"):
        low = read_rgb(low_path)
        high = read_rgb(high_path)
        start = time.perf_counter()
        enhanced = predict_model_np(model, low, DEVICE)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - start) * 1000
        psnr_value, ssim_value = full_reference_metrics(enhanced, high)
        record = {
            "image": low_path.name,
            "method": method_name,
            "psnr": psnr_value,
            "ssim": ssim_value,
            "time_ms": elapsed_ms,
            "niqe": calculate_niqe_optional(enhanced),
        }
        record.update(brightness_metrics(low, enhanced))
        records.append(record)
    df = pd.DataFrame(records)
    if not df.empty:
        display(df.groupby("method")[["psnr", "ssim", "time_ms", "mean_y_gain", "dark_y_gain", "over_exposed_ratio"]].mean())
    return df

## 9. 模型结构自检

这个单元不需要数据集，用随机张量验证模型输入输出尺寸、参数量和 Gamma Map 范围。

In [ ]:
model = build_model("lpia_lightunet").to(DEVICE)
model.eval()
with torch.no_grad():
    dummy = torch.rand(1, 3, 128, 128, device=DEVICE)
    pred, aux = model(dummy, return_aux=True)

print("输出尺寸:", tuple(pred.shape))
print("Gamma Map 范围:", float(aux["gamma_map"].min()), float(aux["gamma_map"].max()))
print(f"参数量: {count_parameters(model) / 1e6:.3f} M")
print(f"模型大小估计: {estimate_model_size_mb(model):.2f} MB")
if DEVICE.type == "cuda":
    print("显存占用 MB:", torch.cuda.max_memory_allocated() / (1024 ** 2))

## 10. 训练主模型

把 LOL-v1 放到 `CFG.data_root` 后，将 `RUN_TRAINING` 改成 `True` 即可开始训练。低算力设备建议先用 `batch_size=2`，跑通后再增大。

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    trained_model, history = fit_model(
        variant="lpia_lightunet",
        epochs=CFG.epochs,
        checkpoint_path=CFG.best_ckpt,
    )
    display(history.tail())
else:
    print("当前未启动训练。将 RUN_TRAINING 改为 True 后运行本单元。")

## 11. 消融实验入口

任务书中的主要消融实验可以通过 `EXPERIMENT_ORDER` 顺序运行。建议课程设计时先把每个模型训练较少 epoch 验证流程，再进行完整训练。

In [ ]:
RUN_ABLATION = False
EXPERIMENT_ORDER = ["plain_unet", "fixed_gamma_unet", "learnable_gamma_unet", "lpia_lightunet"]

if RUN_ABLATION:
    ablation_records = []
    for variant in EXPERIMENT_ORDER:
        print("=" * 80)
        print(variant, EXPERIMENTS[variant])
        ckpt_path = CFG.checkpoint_dir / f"{variant}_best.pt"
        _, hist = fit_model(variant=variant, epochs=CFG.epochs, checkpoint_path=ckpt_path)
        best_row = hist.sort_values("val_psnr", ascending=False).iloc[0].to_dict()
        best_row["variant"] = variant
        best_row["description"] = EXPERIMENTS[variant]
        ablation_records.append(best_row)
    ablation_df = pd.DataFrame(ablation_records)
    ablation_df.to_csv(CFG.output_dir / "ablation_results.csv", index=False)
    display(ablation_df)
else:
    print("当前未启动消融实验。将 RUN_ABLATION 改为 True 后运行本单元。")

## 12. 全参考评价：PSNR / SSIM / 亮度分区 / 推理时间

In [ ]:
RUN_EVALUATION = False
MAX_EVAL_IMAGES = None  # 调试时可设为 3 或 5

if RUN_EVALUATION:
    if not test_pairs:
        raise RuntimeError("未找到测试集，请检查 CFG.data_root。")
    traditional_df = evaluate_traditional_methods(test_pairs, max_images=MAX_EVAL_IMAGES)
    traditional_df.to_csv(CFG.output_dir / "traditional_eval.csv", index=False)
    if CFG.best_ckpt.exists():
        eval_model = load_checkpoint(CFG.best_ckpt)
    else:
        print("没有找到训练好的 checkpoint，使用当前随机初始化模型，仅用于流程检查。")
        eval_model = build_model("lpia_lightunet").to(DEVICE).eval()
    deep_df = evaluate_deep_model(eval_model, test_pairs, max_images=MAX_EVAL_IMAGES)
    deep_df.to_csv(CFG.output_dir / "deep_eval.csv", index=False)
    all_eval_df = pd.concat([traditional_df, deep_df], ignore_index=True)
    all_eval_df.to_csv(CFG.output_dir / "all_eval.csv", index=False)
    display(all_eval_df.groupby("method")[["psnr", "ssim", "time_ms", "mean_y_gain", "dark_y_gain", "over_exposed_ratio"]].mean())
else:
    print("当前未启动评价。将 RUN_EVALUATION 改为 True 后运行本单元。")

## 13. 可视化对比与结果保存

In [ ]:
def compare_one_image(
    low_path: Path,
    high_path: Optional[Path] = None,
    checkpoint_path: Optional[Path] = None,
    save_dir: Optional[Path] = None,
) -> Dict[str, np.ndarray]:
    low = read_rgb(low_path)
    images = {"Low": low}
    for name, fn in TRADITIONAL_METHODS.items():
        images[name] = fn(low)
    if checkpoint_path is not None and Path(checkpoint_path).exists():
        deep_model = load_checkpoint(Path(checkpoint_path))
        images["LPIA-LightU-Net"] = predict_model_np(deep_model, low)
    elif "model" in globals():
        images["LPIA-LightU-Net(untrained)"] = predict_model_np(model, low)
    if high_path is not None and Path(high_path).exists():
        images["Reference"] = read_rgb(high_path)
    show_images(images, cols=min(5, len(images)))
    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        stem = Path(low_path).stem
        for name, img in images.items():
            safe_name = name.replace("/", "_").replace("(", "_").replace(")", "_").replace(" ", "_")
            save_rgb(save_dir / f"{stem}_{safe_name}.png", img)
    return images


# 示例：如果已经找到测试集，可取消注释运行。
# if test_pairs:
#     compare_one_image(test_pairs[0][0], test_pairs[0][1], checkpoint_path=CFG.best_ckpt, save_dir=CFG.result_dir)

## 14. 泛化测试：ExDARK 或 LIME

把无参考低光照图像放到任意文件夹，例如 `datasets/LIME/`，然后修改 `GENERALIZATION_DIR` 并运行。没有参考图时主要看主观视觉、NIQE、亮度提升、过曝比例和推理时间。

In [ ]:
RUN_GENERALIZATION = False
GENERALIZATION_DIR = CFG.project_dir / "datasets" / "LIME"
MAX_GENERALIZATION_IMAGES = 10

if RUN_GENERALIZATION:
    image_paths = list_images(GENERALIZATION_DIR)[:MAX_GENERALIZATION_IMAGES]
    if not image_paths:
        raise RuntimeError(f"未找到泛化测试图像: {GENERALIZATION_DIR}")
    if CFG.best_ckpt.exists():
        gen_model = load_checkpoint(CFG.best_ckpt)
    else:
        print("没有训练好的 checkpoint，使用当前模型，仅用于流程检查。")
        gen_model = build_model("lpia_lightunet").to(DEVICE).eval()
    gen_records = []
    for path in image_paths:
        low = read_rgb(path)
        outputs = {"Low": low}
        for name, fn in TRADITIONAL_METHODS.items():
            outputs[name] = fn(low)
        start = time.perf_counter()
        outputs["LPIA-LightU-Net"] = predict_model_np(gen_model, low)
        elapsed_ms = (time.perf_counter() - start) * 1000
        show_images(outputs)
        for name, img in outputs.items():
            if name == "Low":
                continue
            record = {"image": path.name, "method": name, "time_ms": elapsed_ms if name == "LPIA-LightU-Net" else np.nan}
            record.update(brightness_metrics(low, img))
            record["niqe"] = calculate_niqe_optional(img)
            gen_records.append(record)
            save_rgb(CFG.result_dir / "generalization" / f"{path.stem}_{name}.png", img)
    gen_df = pd.DataFrame(gen_records)
    gen_df.to_csv(CFG.output_dir / "generalization_eval.csv", index=False)
    display(gen_df.groupby("method")[["mean_y_gain", "dark_y_gain", "over_exposed_ratio", "niqe"]].mean())
else:
    print("当前未启动泛化测试。将 RUN_GENERALIZATION 改为 True 后运行本单元。")

## 15. Gradio 可视化系统

支持上传低光照图像、选择增强方法、显示增强结果、推理时间和亮度统计。运行前请确认已安装 `gradio`。

In [ ]:
def enhance_numpy_by_method(image_uint8: np.ndarray, method: str, active_model: Optional[nn.Module] = None) -> Tuple[np.ndarray, Dict[str, float]]:
    image = image_uint8.astype(np.float32) / 255.0
    start = time.perf_counter()
    if method == "Gamma(0.6)":
        enhanced = gamma_correction(image, gamma=0.6)
    elif method == "CLAHE":
        enhanced = clahe_enhance(image)
    elif method == "Retinex":
        enhanced = retinex_enhance(image)
    elif method == "LPIA-LightU-Net":
        if active_model is None:
            raise RuntimeError("未加载模型 checkpoint，请先训练或指定 CFG.best_ckpt。")
        enhanced = predict_model_np(active_model, image)
    else:
        raise ValueError(f"未知方法: {method}")
    elapsed_ms = (time.perf_counter() - start) * 1000
    metrics = brightness_metrics(image, enhanced)
    metrics["time_ms"] = elapsed_ms
    return (np.clip(enhanced * 255.0, 0, 255).astype(np.uint8), metrics)


def build_gradio_app(checkpoint_path: Path = CFG.best_ckpt):
    try:
        import gradio as gr
    except ImportError as exc:
        raise ImportError("请先安装 gradio：%pip install gradio") from exc
    active_model = None
    model_status = "未加载深度模型"
    if Path(checkpoint_path).exists():
        active_model = load_checkpoint(checkpoint_path)
        model_status = f"已加载: {checkpoint_path.name}"
    methods = ["Gamma(0.6)", "CLAHE", "Retinex", "LPIA-LightU-Net"]

    def run_single(image, method):
        if image is None:
            return None, "请先上传图像。"
        try:
            enhanced, metrics = enhance_numpy_by_method(image, method, active_model)
            info = pd.DataFrame([metrics]).T.reset_index()
            info.columns = ["metric", "value"]
            return enhanced, info
        except Exception as exc:
            return None, f"处理失败: {exc}"

    def run_compare(image):
        if image is None:
            return [], "请先上传图像。"
        gallery = []
        rows = []
        for method in methods:
            try:
                enhanced, metrics = enhance_numpy_by_method(image, method, active_model)
                gallery.append((enhanced, method))
                rows.append({"method": method, **metrics})
            except Exception as exc:
                rows.append({"method": method, "error": str(exc)})
        return gallery, pd.DataFrame(rows)

    with gr.Blocks(title="低光照图像增强系统") as demo:
        gr.Markdown("# 低光照图像增强系统")
        gr.Markdown(model_status)
        with gr.Row():
            input_image = gr.Image(label="低光照图像", type="numpy")
            output_image = gr.Image(label="增强结果", type="numpy")
        with gr.Row():
            method_radio = gr.Radio(methods, value="Gamma(0.6)", label="增强方法")
            run_button = gr.Button("运行单方法增强", variant="primary")
            compare_button = gr.Button("多方法对比")
        metrics_table = gr.Dataframe(label="推理时间与亮度统计")
        gallery = gr.Gallery(label="多方法对比", columns=4, height="auto")
        run_button.click(run_single, inputs=[input_image, method_radio], outputs=[output_image, metrics_table])
        compare_button.click(run_compare, inputs=input_image, outputs=[gallery, metrics_table])
    return demo


RUN_GRADIO = False
if RUN_GRADIO:
    demo = build_gradio_app(CFG.best_ckpt)
    demo.launch(server_name="127.0.0.1", server_port=7860)
else:
    print("当前未启动 Gradio。将 RUN_GRADIO 改为 True 后运行本单元。")

## 16. 实验结果整理模板

训练完成后，可以用下面的表格作为报告和结果说明材料 的结果来源。

In [ ]:
def summarize_outputs() -> None:
    files = {
        "训练历史": CFG.history_csv,
        "传统方法评价": CFG.output_dir / "traditional_eval.csv",
        "深度模型评价": CFG.output_dir / "deep_eval.csv",
        "全部评价": CFG.output_dir / "all_eval.csv",
        "消融实验": CFG.output_dir / "ablation_results.csv",
        "泛化评价": CFG.output_dir / "generalization_eval.csv",
    }
    for name, path in files.items():
        print(f"{name}: {path} -> {'存在' if path.exists() else '未生成'}")
    if CFG.best_ckpt.exists():
        ckpt = torch.load(CFG.best_ckpt, map_location="cpu")
        print("最佳模型指标:", ckpt.get("metrics", {}))
    else:
        print("尚未生成最佳模型 checkpoint。")


summarize_outputs()